# Part 2: Student Risk Modeling & Model Comparison
### Project: AI-Powered Agentic Smart Campus Intelligence & Student Success Platform
**Author:** Sumith U (SRN: R24EA807) | 7th Sem B.Tech CSE (AI & ML), REVA University  
**Specification:** Part 2 Student Risk Modeling Workflow  

---

## 1. Objective of Part 2
In this phase, we train and evaluate candidate machine learning algorithms on the anti-leakage prepared dataset from Part 1:
1. **Logistic Regression** (Linear regularized baseline with balanced weights)
2. **Random Forest Classifier** (Bagging tree ensemble)
3. **CatBoost Classifier** (Gradient boosted decision trees with native balance)
4. **XGBoost Classifier** (Cost-sensitive gradient boosting)

We evaluate on the **untouched holdout test set** (885 students) and select the best model based on **Dropout Recall** and **Macro F1**.


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

# Add project root to sys.path
base_dir = Path.cwd().parent
sys.path.insert(0, str(base_dir))

from src.train_model import load_prepared_data, initialize_candidate_models, train_and_evaluate_all
from src.evaluation import evaluate_model_performance

sns.set_theme(style="whitegrid")
%matplotlib inline
print("Part 2 modules and candidate algorithms imported successfully.")


## 2. Load Preprocessed Anti-Leakage Splits
We load `X_train.csv`, `X_test.csv`, and binary dropout targets (`y_train_binary.csv`, `y_test_binary.csv`).


In [ ]:
X_train, X_test, y_train_bin, y_test_bin, y_train_3c, y_test_3c = load_prepared_data()

print(f"Training Features Matrix: {X_train.shape}")
print(f"Testing Features Matrix:  {X_test.shape}")
print(f"Train Class Distribution: Dropout={int((y_train_bin==1).sum())}, Persistent={int((y_train_bin==0).sum())}")
print(f"Test Class Distribution:  Dropout={int((y_test_bin==1).sum())}, Persistent={int((y_test_bin==0).sum())}")


## 3. Train & Evaluate All 4 Candidate Models
We execute 5-Fold Stratified Cross-Validation on `X_train`, fit the models, and evaluate on the untouched `X_test`.


In [ ]:
comparison_df, test_evaluations, best_model_name, best_estimator = train_and_evaluate_all(task="binary")

print("\n=== CANDIDATE ALGORITHMS BENCHMARK SUMMARY ===")
display_cols = ["Model", "CV Macro F1", "Accuracy", "Macro F1", "Dropout Recall", "ROC-AUC", "False Negatives (Missed)"]
comparison_df[display_cols]


## 4. Visual Evaluation: Confusion Matrices & ROC Curves
Let us view the generated confusion matrices and ROC curves comparing the candidate algorithms.


In [ ]:
from IPython.display import Image, display

print("--- Side-by-Side Confusion Matrices ---")
display(Image(filename=str(base_dir / "reports" / "figures" / "confusion_matrix.png")))

print("--- Superimposed ROC Curves ---")
display(Image(filename=str(base_dir / "reports" / "figures" / "roc_curves.png")))


## 5. End-to-End Production Pipeline Verification
We verify that the saved end-to-end pipeline (`models/selected_model_pipeline.joblib`) can take raw, un-preprocessed student records and predict risk.


In [ ]:
pipeline_path = base_dir / "models" / "selected_model_pipeline.joblib"
full_pipeline = joblib.load(pipeline_path)

# Test on 3 raw students from data/raw/data.csv
df_raw = pd.read_csv(base_dir / "data" / "raw" / "data.csv", sep=";").drop(columns=["Target"])
sample_raw = df_raw.iloc[:3]

preds = full_pipeline.predict(sample_raw)
probs = full_pipeline.predict_proba(sample_raw)

print(f"Loaded Pipeline: {full_pipeline}")
for i in range(len(sample_raw)):
    status = "AT-RISK (DROPOUT)" if preds[i] == 1 else "PERSISTENT"
    print(f"Student #{i+1} -> Predicted: {status} | Dropout Probability: {probs[i][1]:.2%}")


## 6. Summary & Preparation for Part 3 (Explainable AI with SHAP)
* **Candidate benchmark completed:** Logistic Regression, Random Forest, CatBoost, and XGBoost.
* **Evidence-based selection:** Saved to `models/selected_model_pipeline.joblib`.
* **Metadata & Report:** Saved to `models/model_metadata.json` and `reports/model_comparison.md`.

**Ready for Part 3: Using SHAP (Shapley Additive Explanations) to explain individual risk drivers.**
